# VeloceReduction — one observing night

**Data flow**

`DetectorFrame → OrderGeometry → OrderMatrix → ExtractionResult`

For `extraction_mode="fibre"`, `FibreGeometry` is an additional Flat-derived input to the final extraction.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from astropy.io import fits
from astropy.table import Table, vstack

from velocereduction import __version__, ReductionConfig
from velocereduction import (
    observations, detector, orders, fibres, flat, extraction,
    calibration, thorium, simlc, wavelength, diagnostics, constants
)
from velocereduction.config import prepare_reduction, setup_logging

night = "001122"
# night = "260703"
config = ReductionConfig(
    night=night,
    extraction_mode="fibre",   # "summed" or "fibre"
    diagnostics="full",
    log_level="DEBUG",
    overwrite=False,
)
paths = prepare_reduction(config, __version__)
logger = setup_logging(config, paths)
print(paths.root)

## 1. Identify observations

The observing log and FITS headers are reconciled first. This stage only decides what data are available and which CCDs should be used.

In [ ]:
reduction_input = observations.identify_observations(config, paths)
display(reduction_input)
print(f"{len(reduction_input)} observations selected for {night}")

## 2. Detector registration

Detector shifts are measured in `detector.py` relative to the reference night. The compact result is written as `detector_shifts_YYMMDD.fits`.


In [ ]:
detector_shifts = detector.measure_detector_shifts(reduction_input, config, paths)
display(detector_shifts)
fits.info(paths.detector_shifts)

## 3. Combine Flats in memory

The individual Flat `DetectorFrame`s are normalized and combined. The full 4112×4096 combined Flat is intentionally **not** saved; it is only an intermediate used to determine geometry and compact 1-D response products.


In [ ]:
combined_flats = flat.combine_flat_frames(reduction_input, config)
for ccd, frame in combined_flats.items():
    print(f"CCD{ccd}: image={frame.image.shape}, finite={np.mean(np.isfinite(frame.image)):.3%}")

## 4. Determine `OrderGeometry`

The reference-night geometry supplies the starting locations. The current Flat and measured detector shifts refine the trace and named cross-dispersion regions. The persistent product has one table row per physical echelle order.


In [ ]:
order_geometry = orders.determine_order_geometry(
    reduction_input, combined_flats, detector_shifts, config, paths
)
order_table = orders.order_geometry_table(order_geometry)
display(order_table[:10])
fits.info(paths.order_geometry)
print(paths.order_geometry)

## 5. Optional compact `FibreGeometry`

For fibre extraction only, the Flat order matrices are fitted at sparse dispersion locations. The saved model contains polynomial coefficients for bundle offset, fibre separation and common Gaussian width, plus one constant offset for each science/sky fibre. The evaluated 4112-row centres are never written to disk.


In [ ]:
flat_order_matrices = flat.extract_flat_order_matrices(combined_flats, order_geometry)
fibre_geometry = {}

if config.extraction_mode == "fibre":
    reference_file = paths.reference_product("fibre_geometry", config.reference_night)
    reference_geometry = fibres.load_fibre_geometry(reference_file) if reference_file.exists() else {}

    if paths.fibre_geometry.exists() and not config.overwrite:
        fibre_geometry = fibres.load_fibre_geometry(paths.fibre_geometry)
    else:
        fibre_geometry = fibres.fit_fibre_geometries(
            flat_order_matrices, config, reference_geometries=reference_geometry
        )
        fibres.save_fibre_geometry(paths.fibre_geometry, fibre_geometry, config)
        fibres.save_fibre_diagnostics(fibre_geometry, flat_order_matrices, config, paths)

    summary = fibres.summarise_fibre_geometry(fibre_geometry)
    display(summary)
    fits.info(paths.fibre_geometry)
    with fits.open(paths.fibre_geometry) as hdul:
        display(Table(hdul["ORDER_MODEL"].data)[:8])
        display(Table(hdul["FIBRE_OFFSETS"].data)[:30])

## 6. Summed and fibre Flat calibrations

For the summed path, the science aperture gives one 4112-pixel Flat spectrum per order; a broad Gaussian-smoothed version is the large-scale illumination/blaze-like shape and their ratio is the small-scale response.

In fibre mode, each extracted fibre is treated independently. `response_fibres` also carries relative fibre-throughput information with respect to the median science-fibre smooth Flat. No 4112×81 multiplicative response map is applied to science/calibration order matrices.


In [ ]:
flat_calibrations = flat.build_flat_calibrations(
    flat_order_matrices, fibre_geometry, config, paths
)

for filename in (
    paths.flat_summed,
    paths.flat_smooth_summed,
    paths.response_summed,
):
    print(filename.name)
    fits.info(filename)

if config.extraction_mode == "fibre":
    for filename in (
        paths.flat_fibres,
        paths.flat_smooth_fibres,
        paths.response_fibres,
    ):
        print(filename.name)
        fits.info(filename)


### Checkpoint: direct summed Flat versus recombined fibres

This is an important independent QA test. The direct summed extraction is retained as the minimally model-dependent reference; wavelength-dependent structure in the fibre/summed ratio can reveal imperfect fibre geometry or deblending.


In [ ]:
if config.extraction_mode == "fibre":
    name = next(name for name in flat_calibrations if name.startswith("ccd_2_"))
    product = flat_calibrations[name]
    geometry = fibre_geometry[name]
    science_components = {str(f) for f in constants.SCIENCE_FIBRES}

    science_idx = np.array([i for i, component in enumerate(geometry.components) if str(component) in science_components], dtype=int)
    recombined = np.nansum(product.fibre_flat[:, science_idx], axis=1)
    scale = np.nanmedian(product.summed_flat / recombined)

    fig, ax = plt.subplots(figsize=(10, 3))
    ax.plot(product.summed_flat, label="direct summed Flat")
    ax.plot(recombined * scale, label="recombined science fibres")
    ax.set(title=name, xlabel="Dispersion pixel", ylabel="Flat counts")
    ax.set_ylim(-1,3)
    ax.legend()
    plt.show()


## 7. Extract calibration spectra in detector coordinates

The same fixed `OrderGeometry` is used for SimTh, SimLC and summed FibTh extraction. Fibre mode additionally extracts the 19 science-fibre FibTh spectra using the fixed `FibreGeometry`. These spectra are deliberately left in detector-pixel coordinates for the next wavelength-calibration stage.


In [ ]:
calibration_exposures = extraction.extract_calibration_exposures(
    reduction_input, order_geometry, fibre_geometry, config
)
calibration_files = extraction.save_calibration_exposures(
    calibration_exposures, paths.calibrations, config.night, overwrite=True
)
for filename in calibration_files:
    print(filename.relative_to(paths.root))


## 8. Wavelength calibration

 Peak measurement and reference-line identification are source-specific (`thorium.py` and `simlc.py`) with `wavelength.py` using the identified line tables to fit the wavelength models.

### 8.1 Reference inputs and bootstrap wavelength solution

The bootstrap solution is used only to identify which laboratory/comb line corresponds to a measured peak. It is not the final nightly wavelength solution. Once the reference night has been validated, `wavelength_static_001122_ccd*.fits` should become the normal bootstrap for subsequent nights.

In [ ]:
Y_BOUNDS = (0.0, 4111.0)
ORDER_BOUNDS = {
    "1": (138, 167),
    "2": (103, 140),
    "3": (65, 104),
}

# Original Murphy atlas + the editable Veloce-specific selection.
murphy_atlas_file = paths.reference_data / "thar_UVES_MM090311.dat"
veloce_atlas_file = paths.reference_data / "veloce_thorium_reference.fits"

if veloce_atlas_file.exists():
    thorium_atlas = thorium.read_veloce_thorium_atlas(veloce_atlas_file)
    print(f"Using curated Veloce Th atlas: {veloce_atlas_file.name}")
else:
    thorium_atlas = thorium.load_murphy_thorium_atlas(murphy_atlas_file)
    print("WARNING: curated Veloce Th atlas not found; using all Murphy Th lines.")

bootstrap_solution = {}
for ccd in ("1", "2", "3"):
    final_reference = (
        paths.reference_data
        / f"wavelength_static_{config.reference_night}_ccd{ccd}.fits"
    )
    legacy_bootstrap = (
        paths.reference_data
        / f"wavelength_bootstrap_{config.reference_night}_ccd{ccd}.fits"
    )
    filename = legacy_bootstrap if legacy_bootstrap.exists() else final_reference
    if not filename.exists():
        raise FileNotFoundError(f"No bootstrap wavelength solution for CCD{ccd}: {filename}")
    bootstrap_solution[ccd], _ = wavelength.read_wavelength_solution_fits(filename)


def detector_shift_y(ccd):
    return detector.detector_shift(detector_shifts, ccd)[1]


def summed_arrays(exposure):
    # extraction stores (4112, n_orders); calibration fitting uses
    # (n_orders, 4112).
    return exposure.summed.flux.T, exposure.summed.variance.T

### 8.2 Calibration-line measurements

#### 8.2.1 FibTh and SimTh

Both thorium sources use the same line measurement/identification machinery. The returned table contains order, measured dispersion position and uncertainty, reference wavelength, quality flags, and detector-space FWHM.

In [ ]:
thorium_line_sets = {
    source: {ccd: [] for ccd in ("1", "2", "3")}
    for source in ("FibTh", "SimTh")
}

# th_to_run = ['FibTh','SimTh']
th_to_run = ['FibTh']

for source in th_to_run:
    for ccd in ("1", "2", "3"):
        for exposure_index, exposure in enumerate(calibration_exposures[source][ccd]):
            filename = (
                paths.calibrations
                / f"{source.lower()}_lines_{config.night}_run{int(exposure.run):04d}_ccd{ccd}.fits"
            )

            if filename.exists() and not config.overwrite:
                try:
                    result = calibration.read_calibration_line_fits(filename)
                    print(f"Read existing {source} line fit: {filename.name}")
                except Exception as error:
                    print(f"Could not read {filename.name} ({error}); remeasuring")
                    result = None
            else:
                result = None

            if result is None:
                counts, variance = summed_arrays(exposure)
                result = thorium.measure_thorium_lines(
                    counts,
                    exposure.orders,
                    thorium_atlas,
                    variance=variance,
                    source=source,
                    ccd=ccd,
                    exposure_index=exposure_index,
                    mjd_mid=exposure.mjd_mid,
                    reference_wavelength_function=bootstrap_solution[ccd].wavelength,
                    detector_shift_y=detector_shift_y(ccd),
                    y_bounds=Y_BOUNDS,
                    minimum_reference_intensity=1.5,
                    diagnostics=config.diagnostics,
                    diagnostic_dir=paths.figures / "calibration",
                    log_level=config.log_level,
                )
                calibration.write_calibration_line_fits(result, filename, overwrite=True)

            thorium_line_sets[source][ccd].append(result)
            n_used = np.count_nonzero(result.lines["used_for_wavelength_fit"])
            print(
                f"{source} CCD{ccd} run {exposure.run}: "
                f"{n_used}/{len(result.lines)} identified lines retained"
            )


#### 8.2.2 SimLC

The laser-frequence comb (LC) provides ~10 times more and well-separated emission peaks with precise peaks for a more precise wavelength solution.  
Here we extract the centroids of these peaks across CCD2 and CCD3:
- First we identify all peaks and fit an integrated Gaussian to extract an initial line-spread-function (LSF) with a specific full-width-half-maximum (FWHM).
- We then parametrise this shape as a smoothly varying LSF.
- We then optimise the centroids with the optimised lsf_shape (with a Moffat shape as default).

In [ ]:
simlc_measurements = {}
source = "SimLC"

for ccd in ('2', '3'):
    for exposure_index, exposure in enumerate(calibration_exposures[source][ccd]):

        filename = (
            paths.calibrations
            / f"simlc_lines_{config.night}_ccd{ccd}.fits"
        )

        if filename.exists() and not config.overwrite:
            try:
                result = calibration.read_calibration_line_fits(filename)
                print(f"Read existing {source} line fit: {filename.name}")
                simlc_measurements[ccd] = result
            except Exception as error:
                print(f"Could not read {filename.name} ({error}); remeasuring")
                result = None
        else:
            result = None

        if result is None:
            simlc_measurements[ccd] = simlc.measure_simlc_lines(
                calibration_exposures[source][ccd][exposure_index],
                paths.reference_data
                / f"wavelength_static_{config.reference_night}_ccd{ccd}.fits",
                lsf_shape="moffat",
                minimum_snr=15,
                diagnostics=config.diagnostics,
                diagnostic_dir=paths.figures,
                output_filename=paths.calibrations
                / f"simlc_lines_{config.night}_ccd{ccd}.fits",
                overwrite=config.overwrite,
                log_level=config.log_level,
            )

### 8.3 Static summed wavelength solution

For the first static solution, choose the FibTh exposure closest to the median calibration time and fit the global $m\lambda(y,m)$ surface. The Legendre degrees are selected by spatially blocked cross-validation rather than from the training residuals.

This is intentionally the **FibTh-only baseline**. The next implementation step is to transform SimLC positions onto the summed-science coordinate and replace the FibTh constraints order-by-order where reliable SimLC coverage exists on CCDs 2 and 3.

In [ ]:
all_calibration_mjds = [
    exposure.mjd_mid
    for source in calibration_exposures.values()
    for ccd_exposures in source.values()
    for exposure in ccd_exposures
    if np.isfinite(exposure.mjd_mid)
]
t0 = float(np.median(all_calibration_mjds))
print(f"Wavelength reference epoch: MJD {t0:.8f}")

static_wavelength = {}
static_fitted_lines = {}
static_validation = {}
static_reference_exposure = {}
static_reference_index = 0

preliminary_solution = {}

for ccd in ("1", "2", "3"):

    for calibration_type in ['FibTh']:#,'SimLC']:#,'FibTh+SimLC']:

        if calibration_type == 'FibTh':

            # FibTh Only
            
            static_reference_exposure[ccd] = calibration_exposures["FibTh"][ccd][static_reference_index]
            fit_input = thorium_line_sets["FibTh"][ccd][static_reference_index].lines
            
        elif calibration_type == 'SimLC':

            if ccd == '1':
                print(f"CCD{ccd}: No SimLC measurements available; skipping")
                continue
            
            # SimLC Only
            fit_input = simlc_measurements[ccd].lines

        elif calibration_type == 'FibTh+SimLC':

            if ccd == '1':
                print(f"CCD{ccd}: No SimLC measurements available; skipping")
                continue

            fit_input, transfer, transferred_lc = wavelength.build_hybrid_static_peak_table(
                thorium_line_sets["FibTh"][ccd][static_reference_index].lines,
                simlc_measurements[ccd].lines,
                preliminary_solution[ccd],
            )

        fit, fitted_lines, validation, chosen = (
            wavelength.fit_validated_wavelength_from_peak_table(
            fit_input,
            y_degrees=[7],
            order_degrees=[5],
            n_folds=5,
            y_blocks=10,
            )
        )

        preliminary_solution[ccd] = fit.solution
        
        static_wavelength[ccd] = fit.solution
        static_fitted_lines[ccd] = fitted_lines
        static_validation[ccd] = validation

        # validation.write(
        #     paths.calibrations / f"wavelength_surface_cv_{calibration_type}_{config.night}_ccd{ccd}.fits",
        #     overwrite=True,
        # )
        # diagnostics.print_wavelength_degree_summary(validation, chosen, ccd=ccd)
        # if config.diagnostics != "none":
        #     diagnostics.save_wavelength_degree_diagnostics(
        #         validation,
        #         chosen,
        #         paths.figures / f"wavelength_surface_cv_{calibration_type}_{config.night}_ccd{ccd}.png",
        #         calibration_type=calibration_type,
        #         ccd=ccd,
        #         diagnostics=config.diagnostics,
        #     )
        wavelength.write_wavelength_fit_fits(
            fit,
            fitted_lines,
            paths.calibrations / f"wavelength_static_{calibration_type}_{config.night}_ccd{ccd}.fits",
            calibration_type=calibration_type,
            ccd=ccd,
            mjd_mid=static_reference_exposure[ccd].mjd_mid,
            # source_peak_file=(
            #     f"fibth_lines_{config.night}_run{int(exposure.run):04d}_ccd{ccd}.fits"
            # ),
            overwrite=True,
        )

        diagnostics.print_wavelength_fit_summary(
            fit, fitted_lines, calibration_type=calibration_type, ccd=ccd,
        )
        if config.diagnostics != "none":
            diagnostics.save_wavelength_diagnostics(
                fit,
                fitted_lines,
                paths.figures / f"wavelength_static_{calibration_type}_{config.night}_ccd{ccd}.png",
                calibration_type=calibration_type,
                ccd=ccd,
                diagnostics=config.diagnostics,
            )

### 8.4 Joint fibre displacement reference

In fibre mode the first FibTh exposure is used to load or construct the joint rank-2 fibre displacement model. Per-fibre line measurement, matching, fitting, reference FITS I/O, and QA plotting are handled inside `wavelength.ensure_fibre_displacement_model()`.


In [ ]:
fibre_displacement_model = None

if config.extraction_mode == "fibre":
    fibre_displacement_model, matched_fibre_lines = wavelength.ensure_fibre_displacement_model(
        config=config,
        paths=paths,
        calibration_exposures=calibration_exposures,
        summed_fibth_line_sets={
            ccd: thorium_line_sets["FibTh"][ccd][static_reference_index]
            for ccd in ("1", "2", "3")
        },
        static_wavelength=static_wavelength,
        thorium_atlas=thorium_atlas,
        static_fibth_reference_index=static_reference_index,
    )

    print(
        "Fibre displacement reference: "
        f"{fibre_displacement_model.reference_night or config.reference_night}; "
        f"rank={fibre_displacement_model.rank}, degree={fibre_displacement_model.degree}"
    )


In [ ]:
if config.extraction_mode == "fibre" and config.diagnostics != "none":
    print("Fibre displacement before/after QA written to", paths.figures / "calibration")


### 8.5 Temporal wavelength corrections

All SimTh and SimLC exposures have already been reduced to homogeneous line tables above. The forthcoming temporal model will compare each source against its own reference epoch so that the fixed SimTh/SimLC fibre offset is not confused with temporal drift.

In [ ]:
time_shift_model = {ccd: None for ccd in ("1", "2", "3")}
time_shift_qa = {}
time_shift_series = {}
wavelength_model = {}

for ccd in static_wavelength:
    lc_sets = simlc_measurements.get(ccd, [])
    th_sets = thorium_line_sets["SimTh"].get(ccd, [])

    try:
        time_model, qa, all_series = wavelength.fit_time_corrections(
            static_solution=static_wavelength[ccd],
            simlc_line_sets=lc_sets,
            simth_line_sets=th_sets,
            reference_mjd=t0,
            y_degree=2,
            order_degree=1,
            preferred_source="SimLC",
        )
        time_shift_model[ccd] = time_model
        time_shift_qa[ccd] = qa
        time_shift_series[ccd] = all_series
        qa.write(
            paths.calibrations / f"wavelength_time_qa_{config.night}_ccd{ccd}.fits",
            overwrite=True,
        )
        print(
            f"CCD{ccd}: temporal source={time_model.source}, "
            f"N={len(time_model.mjd)}, reference MJD={time_model.reference_mjd:.8f}"
        )
    except RuntimeError as error:
        print(f"CCD{ccd}: no temporal correction ({error})")

    wavelength_model[ccd] = wavelength.WavelengthCalibration(
        static=static_wavelength[ccd],
        fibre=fibre_displacement_model,
        time=time_shift_model.get(ccd),
        ccd=ccd,
    )
    wavelength.write_wavelength_model_fits(
        wavelength_model[ccd],
        paths.calibrations / f"wavelength_model_{config.night}_ccd{ccd}.fits",
        ccd=ccd,
        reference_mjd=t0,
        overwrite=True,
    )


### 8.6 Compact QA / usage check

In [ ]:
for ccd, model in wavelength_model.items():
    m0 = int(np.round(np.mean(ORDER_BOUNDS[ccd])))
    y0 = 0.5 * sum(Y_BOUNDS)
    message = [f"CCD{ccd}: lambda({y0:.1f}, m={m0})={model.static.wavelength(y0, m0):.6f} nm"]
    if model.time is not None:
        drift = [float(model.time.shift(y0, m0, t)) for t in model.time.mjd]
        message.append(
            f"{model.time.source} drift range=[{np.min(drift):+.4f}, {np.max(drift):+.4f}] pix"
        )
    print("; ".join(message))

# Example for a science/fibre spectrum:
# y = np.arange(4112, dtype=float)
# wave_nm = wavelength_model[ccd].wavelength(
#     y, order, fibre=int(fibre_name), mjd=science_mjd_mid
# )

### 9 Resolution profile

The peak tables persist `fwhm_pixel` and `fwhm_uncertainty_pixel`. Once the final wavelength model is fixed, the local wavelength FWHM and resolving power can be derived from

$$
\Delta\lambda_{\rm FWHM}
=
\left|\frac{d\lambda}{dy}\right|
{\rm FWHM}_{\rm pix},
\qquad
\mathcal{R}
=
\frac{\lambda}{\Delta\lambda_{\rm FWHM}}.
$$

The smooth $\mathcal{R}(y,m)$ model should therefore be fitted after the wavelength solution rather than stored as a primary line-fit quantity.

In [ ]:
# resolution_measurements = {}

# for ccd, lines in static_fitted_lines.items():
#     used = np.asarray(lines["used_for_wavelength_fit"], bool)
#     y = np.asarray(lines["y"], float)
#     order = np.asarray(lines["order"], int)
#     fwhm_pixel = np.asarray(lines["fwhm_pixel"], float)

#     wavelength_nm = static_wavelength[ccd].wavelength(y, order)
#     dispersion_nm_per_pixel = np.abs(
#         static_wavelength[ccd].dispersion(y, order)
#     )
#     delta_lambda_nm = dispersion_nm_per_pixel * fwhm_pixel
#     resolving_power = np.divide(
#         wavelength_nm,
#         delta_lambda_nm,
#         out=np.full_like(wavelength_nm, np.nan),
#         where=np.isfinite(delta_lambda_nm) & (delta_lambda_nm > 0),
#     )

#     resolution_measurements[ccd] = Table(
#         {
#             "order": order[used],
#             "y": y[used],
#             "wavelength_nm": wavelength_nm[used],
#             "fwhm_pixel": fwhm_pixel[used],
#             "resolving_power": resolving_power[used],
#         }
#     )

#     display(resolution_measurements[ccd][:5])

# # NEXT: fit a smooth resolution profile in (y, m).

## 9. Science extraction


In [ ]:
# This needs to be updated later, once we have all the calibration products, to extract and calibrate thescience exposures

# science_exposures = science.extract_science_exposures(
#     reduction_input, nightly_tramlines, flat_products,
#     wavelength_model, config, paths
# )
# print(f"{len(science_exposures)} science CCD exposures")

# if science_exposures:
#     exposure = science_exposures[0]
#     order = exposure.orders[len(exposure.orders) // 2]
#     plt.figure(figsize=(10, 3))
#     plt.plot(order.barycentric_wavelength_nm, order.flux)
#     plt.xlabel("Barycentric wavelength / nm")
#     plt.ylabel("Flux")
#     plt.title(f"{exposure.object_name} — CCD{exposure.ccd}, order {order.order}")
#     plt.show()


## One-call equivalent


In [ ]:
# from velocereduction import pipeline
# state = pipeline.reduce_night(config, version=__version__)
